In [9]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAI
from dotenv import load_dotenv


In [10]:
load_dotenv()

True

In [11]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

In [12]:
class LLMState(TypedDict):
    question:str
    answer:str

In [13]:
def llm_qa(state:LLMState)->LLMState:
    question=state['question']
    prompt=f'Answer the following question {question}'
    answer=llm.invoke(prompt).text
    state['answer']=answer
    
    return state


In [18]:
graph=StateGraph(LLMState)
graph.add_node('llm_qa',llm_qa)
graph.add_edge(START,"llm_qa")
graph.add_edge('llm_qa',END)
workflow=graph.compile()

In [21]:
initial_state={'question':'What is langgraph ?'}
final_state=workflow.invoke(initial_state)
print(final_state['answer'])

**LangGraph** is an open-source framework developed by the creators of LangChain. It is designed for building **stateful, multi-actor applications with Large Language Models (LLMs)**. 

While LangChain is excellent for creating linear, step-by-step chains (e.g., Input $\rightarrow$ Prompt $\rightarrow$ LLM $\rightarrow$ Output), LangGraph is built specifically to handle **complex, circular workflows (cycles)**, which are essential for building advanced AI agents.

Here is a detailed breakdown of what LangGraph is, how it works, and why it is important.

---

### 1. The Core Metaphor: Graphs
LangGraph models LLM applications as a **graph** (specifically, a stateful graph). It consists of three main components:

*   **State:** This is the shared memory of your application. Every step in the graph can read from and write to this state. It keeps track of the conversation history, retrieved documents, or variables.
*   **Nodes:** These represent steps of execution. A node is typically a Pyt